# AI Commerce Intelligence Platform
## Notebook 02: Demand Intelligence, Stockout Masking & Feature Store

**Objective**: Reconstruct regularized daily demand across SKU-warehouse combinations, isolate periods where inventory constraints capped customer sales, model demand intermittency, and assemble leakage-free feature pipelines.

### Business Context: Why Demand ≠ Sales
In ecommerce and omnichannel operations, **sales** measure what the warehouse was physically able to ship, whereas **demand** measures what customers actually intended to buy.
When a product runs out of stock, recorded sales drop to zero. A forecasting engine trained on uncorrected sales data would mistake inventory failure for lack of market interest, leading to smaller replenishment orders and perpetual stockouts.


In [ ]:
import os
import sys
from pathlib import Path

workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "notebooks":
    workspace_dir = workspace_dir.parent
sys.path.insert(0, str(workspace_dir / "src"))

# Matplotlib cache in workspace
mpl_cache = workspace_dir / ".gemini" / "scratch" / "mpl"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_cache)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from commerce_ai.data.loaders import CommerceDataLoader
from commerce_ai.analytics.demand import build_daily_demand
from commerce_ai.analytics.stockout import StockoutConfig, detect_stockout_periods, mask_stockout_demand
from commerce_ai.analytics.abc_xyz import calculate_abc_xyz_matrix
from commerce_ai.analytics.features import (
    calculate_intermittency_metrics,
    build_lag_features,
    build_rolling_features,
    build_trend_features,
    build_demand_features,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10


### 1. Ingest Canonical Ecommerce Datasets
Loading verified sample data generated in Phase 1.

In [ ]:
data_dir = workspace_dir / "data" / "sample"
loader = CommerceDataLoader()
datasets = loader.load_all(data_dir, validate=False, validate_relational=False)

sales_df = datasets["sales"]
inventory_df = datasets["inventory"]
products_df = datasets["products"]
warehouses_df = datasets["warehouses"]

print(f"Loaded {len(sales_df):,d} sales transactions and {len(inventory_df):,d} inventory snapshots.")


### 2. Daily SKU-Warehouse Demand Grid
Reconstructing continuous daily demand time series for representative high and medium velocity SKUs across fulfillment nodes.
Notice that days with zero sales are explicitly preserved as `units_sold = 0` rather than omitted.

In [ ]:
# Select representative portfolio (top 20 SKUs by volume across warehouses)
top_skus = sales_df.groupby("sku_id")["quantity"].sum().nlargest(20).index.tolist()

daily_demand = build_daily_demand(
    sales=sales_df,
    inventory=inventory_df,
    products=products_df,
    warehouses=warehouses_df,
    skus=top_skus,
)

print(f"Reconstructed daily demand grid: {len(daily_demand):,d} rows")
print(f"Dates: {daily_demand['date'].min().date()} to {daily_demand['date'].max().date()}")
print(f"Unique SKUs: {daily_demand['sku_id'].nunique()} across {daily_demand['warehouse_id'].nunique()} Warehouses")
daily_demand.head(5)


### 3. Aggregate Demand Velocity & Category Trends
Tracking rolling daily units sold and gross revenue over time.

In [ ]:
trend_df = daily_demand.groupby("date").agg(
    total_units=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    active_pairs=("units_sold", lambda s: (s > 0).sum())
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(trend_df["date"], trend_df["total_units"], label="Daily Units", color="steelblue", alpha=0.5)
ax1.plot(trend_df["date"], trend_df["total_units"].rolling(14).mean(), label="14-Day Rolling Avg", color="navy", linewidth=2)
ax1.set_ylabel("Units Sold", color="navy")
ax1.set_xlabel("Date")
ax1.set_title("Reconstructed Daily Demand Velocity (20-SKU Core Portfolio)")
ax1.legend(loc="upper left")
plt.tight_layout()
plt.show()


### 4. Stockout Detection & Running Event Analysis
Identifying periods where inventory depletion halted order fulfillment.

In [ ]:
stockout_cfg = StockoutConfig(stockout_threshold=0, low_stock_threshold=5)
daily_demand_stockout = detect_stockout_periods(daily_demand, config=stockout_cfg)

total_obs = len(daily_demand_stockout)
stockout_obs = daily_demand_stockout["is_stockout"].sum()
low_stock_obs = daily_demand_stockout["is_low_stock"].sum()

print(f"Total Daily Observations: {total_obs:,d}")
print(f"Stockout Observations:   {stockout_obs:,d} ({stockout_obs / total_obs * 100:.2f}%)")
print(f"Low Stock Observations:   {low_stock_obs:,d} ({low_stock_obs / total_obs * 100:.2f}%)")

# Distribution of stockout duration (days in stockout)
active_stockout_days = daily_demand_stockout[daily_demand_stockout["is_stockout"]]["stockout_days"]
print(f"Average Stockout Duration: {active_stockout_days.mean():.1f} consecutive days (Max: {active_stockout_days.max()} days)")


### 5. Constrained vs. Unconstrained Demand Masking
Visualizing an actual stockout event: during stockouts, sales collapse to zero because on-hand stock is 0.
The `forecast_training_eligible` flag cleanly isolates these observations so training datasets reflect true demand.

In [ ]:
masked_demand = mask_stockout_demand(daily_demand_stockout, config=stockout_cfg)

# Find SKU with significant stockout days
so_sku = masked_demand[masked_demand["is_stockout"]]["sku_id"].value_counts().index[0]
sample_wh = masked_demand[masked_demand["sku_id"] == so_sku]["warehouse_id"].iloc[0]
sku_wh_series = masked_demand[(masked_demand["sku_id"] == so_sku) & (masked_demand["warehouse_id"] == sample_wh)].sort_values("date")

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(sku_wh_series["date"], sku_wh_series["available_qty"], color="seagreen", label="Available Inventory", linewidth=2)
ax1.axhline(0, color="crimson", linestyle="--", alpha=0.7, label="Stockout Floor (0 Units)")
ax1.set_ylabel("Available Units on Hand", color="seagreen")
ax1.set_xlabel("Date")

ax2 = ax1.twinx()
clean_sales = sku_wh_series[sku_wh_series["forecast_training_eligible"]]
masked_sales = sku_wh_series[~sku_wh_series["forecast_training_eligible"]]

ax2.scatter(clean_sales["date"], clean_sales["units_sold"], color="royalblue", alpha=0.6, s=15, label="Clean Demand (Eligible)")
ax2.scatter(masked_sales["date"], masked_sales["units_sold"], color="crimson", marker="x", s=25, label="Masked Demand (Stockout)")
ax2.set_ylabel("Daily Units Sold", color="royalblue")

plt.title(f"Demand Masking Case Study: SKU {so_sku} at {sample_wh}")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()


### 6. Top SKUs by Revenue Contribution
Auditing revenue generation across the portfolio.

In [ ]:
sku_revenue = sales_df.groupby("sku_id")["revenue"].sum().sort_values(ascending=False).reset_index()
sku_revenue["revenue_share"] = sku_revenue["revenue"] / sku_revenue["revenue"].sum()
sku_revenue["cum_share"] = sku_revenue["revenue_share"].cumsum()

top_10 = sku_revenue.head(10).merge(products_df[["sku_id", "product_name", "category_id"]], on="sku_id")
print("Top 10 Revenue Generators:")
display_cols = ["sku_id", "product_name", "category_id", "revenue", "revenue_share", "cum_share"]
top_10[display_cols].style.format({"revenue": "${:,.2f}", "revenue_share": "{:.2%}", "cum_share": "{:.2%}"})


### 7. Demand Variability & Coefficient of Variation
Comparing CV calculated on raw sales versus unconstrained (stockout-masked) demand.

In [ ]:
# Compute CV raw vs masked
raw_cv = masked_demand.groupby("sku_id")["units_sold"].agg(mean="mean", std="std")
raw_cv["cv_raw"] = raw_cv["std"] / raw_cv["mean"].replace(0, np.nan)

clean_subset = masked_demand[masked_demand["forecast_training_eligible"]]
clean_cv = clean_subset.groupby("sku_id")["units_sold"].agg(mean="mean", std="std")
clean_cv["cv_unconstrained"] = clean_cv["std"] / clean_cv["mean"].replace(0, np.nan)

comparison = pd.merge(raw_cv[["cv_raw"]], clean_cv[["cv_unconstrained"]], on="sku_id").dropna()

fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(x="cv_raw", y="cv_unconstrained", data=comparison, color="darkslateblue", s=60, ax=ax)
ax.plot([0, 2], [0, 2], color="gray", linestyle="--", label="y = x (Equality)")
ax.set_title("Demand Volatility: Raw CV vs. Stockout-Masked CV")
ax.set_xlabel("Raw Sales CV (Includes Artificial Zeroes)")
ax.set_ylabel("Unconstrained CV (Clean Demand)")
ax.legend()
plt.tight_layout()
plt.show()
print("Notice: Masking stockouts eliminates artificial zero-demand variance, providing a true measure of consumer volatility.")


### 8. Demand Intermittency Classification
Segmenting time series into Regular, Intermittent, and Highly Intermittent based on occurrence frequency.

In [ ]:
intermittency_df = calculate_intermittency_metrics(daily_demand, target_col="units_sold")
print("Intermittency Distribution:")
print(intermittency_df["intermittency_class"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(intermittency_df["demand_occurrence_rate"], bins=15, kde=True, ax=axes[0], color="teal")
axes[0].set_title("Distribution of Demand Occurrence Rate")
axes[0].set_xlabel("Proportion of Days with Active Sales")

sns.countplot(x="intermittency_class", hue="intermittency_class", data=intermittency_df, ax=axes[1], palette="crest", legend=False)
axes[1].set_title("Portfolio Breakdown by Demand Archetype")
axes[1].set_xlabel("Intermittency Archetype")
plt.tight_layout()
plt.show()


### 9. Demand Probability Distributions
Analyzing non-zero demand density distributions across categories.

In [ ]:
demand_with_cat = daily_demand[daily_demand["units_sold"] > 0].merge(
    products_df[["sku_id", "category_id"]], on="sku_id"
)

plt.figure(figsize=(12, 4))
sns.boxplot(x="category_id", y="units_sold", hue="category_id", data=demand_with_cat, palette="Set2", legend=False)
plt.yscale("log")
plt.title("Active Daily Sales Volume by Category (Log Scale)")
plt.xlabel("Category")
plt.ylabel("Units Sold per Day")
plt.tight_layout()
plt.show()


### 10. Master Demand Feature Store Dataset
Executing `build_demand_features` to assemble the production-ready feature matrix.

In [ ]:
# Build full feature set on the representative portfolio
features_dataset = build_demand_features(daily_demand)

print(f"Generated Feature Matrix: {features_dataset.shape[0]:,d} rows × {features_dataset.shape[1]:,d} columns")
print(f"Features Generated:\n{list(features_dataset.columns)}")
features_dataset.head(5)


### Key Takeaways for Product & Forecasting
1. **Demand Reconstruction**: Creating the full regularized grid prevents forecasting algorithms from silently dropping non-sales days.
2. **Causal Leakage Protection**: All rolling statistics operate on causally shifted series (`shift(1)`), guaranteeing zero forward leakage.
3. **Causal Stockout Masking**: Distinguishing `forecast_training_eligible` prevents zero-demand distortions during stockouts.
4. **Readiness**: The feature dataset is primed for Phase 3 Time-Series models (LightGBM, TimesFM-3).
